# 02_loss_gradient_check.ipynb
=======================================
Loss Function Sanity Check Notebook.

Run this BEFORE full training to verify:
1. All three loss components produce valid (non-NaN) gradients
2. Heterogeneity weights correctly modulate gradient magnitudes
3. Relative error term dominates over NB likelihood for large errors
4. Focal loss down-weights easy negatives as expected
5. Partial baseline outputs (None threat/admin) don't crash composite score
"""

# Cell 1: Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import numpy as np
from src.losses.hybrid_loss import HybridHeterogeneousLoss, ZeroInflatedNBRelativeLoss, FocalBCELoss
from src.utils.metrics import compute_composite_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


# Cell 2: Test Zero-Inflated NB + Relative Error Loss

In [ ]:
print("=" * 60)
print("TEST 1: Zero-Inflated NB + Relative Error Loss")
print("=" * 60)

nb_loss_fn = ZeroInflatedNBRelativeLoss(beta=1.0).to(device)

# Simulate predictions and targets [B=2, F=1, N=4, C=3]
mu = torch.tensor([[[[2.0, 0.5, 10.0],
                      [0.1, 5.0, 0.3],
                      [8.0, 1.0, 0.2],
                      [0.0, 0.0, 0.0]]]], device=device, requires_grad=True)
alpha = torch.ones_like(mu, requires_grad=True) * 2.0
target = torch.tensor([[[[3, 0, 12],
                          [0, 4, 0],
                          [10, 0, 0],
                          [0, 0, 0]]]], device=device, dtype=torch.float32)

loss = nb_loss_fn(mu, alpha, target)
loss.backward()

print(f"  Loss value: {loss.item():.4f}")
print(f"  mu grad NaN: {torch.isnan(mu.grad).any().item()}")
print(f"  mu grad norm: {mu.grad.norm().item():.6f}")
print(f"  alpha grad NaN: {torch.isnan(alpha.grad).any().item()}")

# Verify relative error dominance: element [0,0,0,2] has truth=12, pred=10 → 16.7% rel error
# Element [0,0,0,0] has truth=3, pred=2 → 33.3% rel error
# The gradient for [0,0,0,0] should be larger due to higher relative error
rel_err_grad_ratio = mu.grad[0,0,0,0].abs() / (mu.grad[0,0,0,2].abs() + 1e-8)
print(f"  RelError gradient ratio (high-rel / low-rel): {rel_err_grad_ratio.item():.2f} (should be > 1.0)")
assert not torch.isnan(loss), " FAIL: NB loss is NaN!"
assert not torch.isnan(mu.grad).any(), " FAIL: mu gradient contains NaN!"
print("   PASS\n")

# Cell 3: Test Focal BCE Loss

In [ ]:
print("=" * 60)
print("TEST 2: Focal BCE Loss")
print("=" * 60)

focal_loss_fn = FocalBCELoss(gamma=2.0).to(device)

# Simulate: mostly easy negatives, one rare positive
pred_probs = torch.tensor([[0.01, 0.95, 0.02, 0.03]], device=device, requires_grad=True)
target_binary = torch.tensor([[0.0, 1.0, 0.0, 0.0]], device=device)

focal_loss = focal_loss_fn(pred_probs, target_binary)
focal_loss.backward()

print(f"  Loss value: {focal_loss.item():.6f}")
print(f"  Grad NaN: {torch.isnan(pred_probs.grad).any().item()}")

# Easy negative (index 0, pred=0.01, target=0) should have smaller gradient than
# hard positive (index 1, pred=0.95, target=1) — focal loss up-weights hard examples
easy_neg_grad = pred_probs.grad[0, 0].abs()
hard_pos_grad = pred_probs.grad[0, 1].abs()
print(f"  Easy neg grad: {easy_neg_grad.item():.6f}")
print(f"  Hard pos grad: {hard_pos_grad.item():.6f}")
print(f"  Ratio (hard/easy): {(hard_pos_grad / (easy_neg_grad + 1e-8)).item():.2f} (should be >> 1.0)")
assert not torch.isnan(focal_loss), "FAIL: Focal loss is NaN!"
print(" PASS\n")

# Cell 4: Test Heterogeneity Weight Modulation

In [ ]:


print("=" * 60)
print("TEST 3: Heterogeneity Weight Gradient Modulation")
print("=" * 60)

hybrid_loss = HybridHeterogeneousLoss().to(device)

# Create synthetic count tensor with heterogeneous grids
# Grid 0: low variance (steady), Grid 1: high variance (bursty)
X_train = torch.zeros(100, 4, 3)  # [T, N, C]
X_train[:, 0, :] = 2.0  # Grid 0: constant → low Fano factor
X_train[:, 1, :] = torch.poisson(torch.tensor(5.0)).float()  # Grid 1: bursty → high Fano factor
X_train[:, 2, :] = 0.0  # Grid 2: always zero → edge case

weights = hybrid_loss.compute_heterogeneity_weights(X_train.to(device))
print(f"  Weights: {weights.cpu().numpy()}")
print(f"  Grid 0 (steady) weight: {weights[0].item():.3f}")
print(f"  Grid 1 (bursty) weight: {weights[1].item():.3f}")
print(f"  Grid 2 (zero) weight:   {weights[2].item():.3f}")
assert weights[1] > weights[0], " FAIL: Bursty grid should have higher weight than steady grid!"
print("   PASS: Heterogeneity weights correctly prioritize bursty grids\n")


# Cell 5: Test Partial Baseline Output Compatibility

In [ ]:
print("=" * 60)
print("TEST 4: Partial Baseline Output (No Threat/Admin Heads)")
print("=" * 60)

pred_counts = torch.rand(4, 1, 10, 58)
tgt_counts = torch.poisson(torch.tensor(2.0)).expand_as(pred_counts).float()

# Simulate baseline with NO threat head
metrics = compute_composite_score(
    pred_counts=pred_counts,
    target_counts=tgt_counts,
    pred_threat=None,       # ← Baseline doesn't produce this
    target_threat=None,     # ← Baseline doesn't produce this
    hetero_weights=weights[:10],
    baseline_name="test_baseline"
)

print(f"  Composite score: {metrics['composite_score']:.4f}")
print(f"  Threat status: {metrics['threat_status']}")
print(f"  Model tag: {metrics.get('model', 'MISSING')}")
assert metrics['threat_status'] == 'excluded_no_threat_head', " FAIL: Should flag missing threat head!"
assert metrics['model'] == 'test_baseline', " FAIL: Missing baseline name tag!"
assert not np.isnan(metrics['composite_score']), " FAIL: Composite score is NaN with partial output!"
print("   PASS: Partial outputs handled gracefully\n")

# Cell 6: Full Hybrid Loss Integration Test

In [ ]:
print("=" * 60)
print("TEST 5: Full Hybrid Loss Integration")
print("=" * 60)

predictions = {
    'count_mu': torch.rand(2, 1, 4, 58, device=device, requires_grad=True) * 5,
    'count_alpha': torch.ones(2, 1, 4, 58, device=device, requires_grad=True) * 2,
    'admin_logits': torch.randn(2, 1, 4, 4, device=device, requires_grad=True),
    'threat_probs': torch.rand(2, 1, 4, 9, device=device, requires_grad=True),
}
targets = {
    'y_count': torch.poisson(torch.tensor(2.0)).expand(2, 1, 4, 58).float().to(device),
    'y_admin': torch.zeros(2, 1, 4, 4, device=device).scatter_(-1, 
               torch.randint(0, 4, (2, 1, 4, 1), device=device), 1.0),
    'y_threat': (torch.rand(2, 1, 4, 9, device=device) > 0.9).float(),
}

loss_dict = hybrid_loss(predictions, targets)
loss_dict['total_loss'].backward()

print(f"  Total loss: {loss_dict['total_loss'].item():.4f}")
print(f"  Count loss: {loss_dict['count_loss'].item():.4f}")
print(f"  Admin loss: {loss_dict['admin_loss'].item():.4f}")
print(f"  Threat loss: {loss_dict['threat_loss'].item():.4f}")

all_grads_valid = all(
    not torch.isnan(p.grad).any()
    for p in [predictions['count_mu'], predictions['count_alpha'],
              predictions['admin_logits'], predictions['threat_probs']]
    if p.grad is not None
)
print(f"  All gradients valid: {all_grads_valid}")
assert all_grads_valid, " FAIL: Some gradients contain NaN!"
print("   PASS\n")

print("=" * 60)
print("ALL TESTS PASSED  — Loss function is ready for training")
print("=" * 60)